<a href="https://colab.research.google.com/github/Deepakgg/driver-drowsiness-detection/blob/main/Driver_drowsiness_detection_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
#  Driver Drowsiness Detection System
#  Based on: MediaPipe FaceMesh + EAR Algorithm
#
#  Architecture (from project spec):
#  Webcam → BGR→RGB → MediaPipe FaceMesh → Eye Landmarks
#           → EAR Calc → Frame Counter Logic → Alarm Alert
#
#  HOW TO RUN
#  ----------
#  1. pip install opencv-python mediapipe numpy playsound
#  2. Place alarm.wav in the same folder (optional)
#  3. python drowsiness_detector.py
#  4. Press 'q' to quit
# ============================================================

import cv2
import mediapipe as mp
import numpy as np
import threading
import time
import os
import sys

# Try importing playsound — gracefully fall back if missing
try:
    from playsound import playsound
    PLAYSOUND_AVAILABLE = True
except ImportError:
    PLAYSOUND_AVAILABLE = False
    print("[WARN] playsound not found. Run: pip install playsound")
    print("       Alarm will print to terminal instead.")


# ─────────────────────────────────────────────────────────────
#  CONFIGURATION  (edit these to tune the system)
# ─────────────────────────────────────────────────────────────

EAR_THRESHOLD  = 0.25   # below this → eye is closing
FRAME_LIMIT    = 15     # consecutive frames below threshold → ALERT
ALARM_SOUND    = "alarm.wav"


# ─────────────────────────────────────────────────────────────
#  MediaPipe FaceMesh landmark indices for the 6 eye points
#  Source: project specification (page 5)
#
#         P2  P3
#        /      \
#   P1 ————————— P4
#        \      /
#         P6  P5
#
#  Eye      P1   P2   P3   P4   P5   P6
#  Left     33  160  158  133  153  144
#  Right   362  385  387  263  373  380
# ─────────────────────────────────────────────────────────────

LEFT_EYE  = [33,  160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]


# ─────────────────────────────────────────────────────────────
#  HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────

def compute_ear(landmarks, eye_indices, img_w, img_h):
    """
    Compute Eye Aspect Ratio (EAR) for one eye.

    Formula:  EAR = ( ||P2-P6|| + ||P3-P5|| ) / ( 2 × ||P1-P4|| )

    landmarks   : MediaPipe NormalizedLandmarkList
    eye_indices : list of 6 landmark indices [P1..P6]
    img_w/h     : frame dimensions (to convert normalised → pixels)
    """
    # Extract (x, y) pixel coords for the 6 points
    pts = []
    for idx in eye_indices:
        lm = landmarks[idx]
        pts.append((lm.x * img_w, lm.y * img_h))

    P1, P2, P3, P4, P5, P6 = pts

    # Vertical distances
    vertical_1 = np.linalg.norm(np.array(P2) - np.array(P6))
    vertical_2 = np.linalg.norm(np.array(P3) - np.array(P5))

    # Horizontal distance
    horizontal = np.linalg.norm(np.array(P1) - np.array(P4))

    ear = (vertical_1 + vertical_2) / (2.0 * horizontal)
    return ear, pts


def draw_eye_outline(frame, pts, color):
    """Draw lines connecting the 6 eye landmark points."""
    pts_array = np.array(pts, dtype=np.int32)
    hull = cv2.convexHull(pts_array)
    cv2.drawContours(frame, [hull], -1, color, 1)
    for (x, y) in pts:
        cv2.circle(frame, (int(x), int(y)), 2, color, -1)


def sound_alarm():
    """Play alarm sound in a non-blocking thread."""
    if PLAYSOUND_AVAILABLE and os.path.exists(ALARM_SOUND):
        try:
            playsound(ALARM_SOUND)
        except Exception as e:
            print(f"[WARN] Could not play alarm: {e}")
    else:
        # Terminal bell fallback
        print("\a*** DROWSINESS ALERT! Wake up! ***")


def draw_status_bar(frame, ear, frame_counter, alarm_on):
    """Draw a transparent HUD at the bottom of the frame."""
    h, w = frame.shape[:2]
    overlay = frame.copy()

    # Dark bar at bottom
    cv2.rectangle(overlay, (0, h - 75), (w, h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    # EAR value
    ear_color = (0, 255, 0) if ear > EAR_THRESHOLD else (0, 0, 255)
    cv2.putText(frame, f"EAR: {ear:.3f}",
                (15, h - 48), cv2.FONT_HERSHEY_SIMPLEX, 0.6, ear_color, 2)

    # Threshold line indicator
    cv2.putText(frame, f"Threshold: {EAR_THRESHOLD}",
                (15, h - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)

    # Frame counter bar
    bar_x, bar_y = 200, h - 55
    bar_w = 220
    filled = int((frame_counter / FRAME_LIMIT) * bar_w)
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + 18), (60, 60, 60), -1)
    bar_color = (0, 200, 255) if frame_counter < FRAME_LIMIT else (0, 0, 255)
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + filled, bar_y + 18), bar_color, -1)
    cv2.putText(frame, f"Frames: {frame_counter}/{FRAME_LIMIT}",
                (bar_x + 5, bar_y + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

    # Status text
    status_text  = "ALARM ACTIVE" if alarm_on else ("Eyes closing..." if frame_counter > 0 else "Normal")
    status_color = (0, 0, 255) if alarm_on else ((0, 165, 255) if frame_counter > 0 else (0, 255, 100))
    cv2.putText(frame, status_text,
                (450, h - 35), cv2.FONT_HERSHEY_SIMPLEX, 0.65, status_color, 2)


# ─────────────────────────────────────────────────────────────
#  INITIALISE MediaPipe FaceMesh
# ─────────────────────────────────────────────────────────────

print("[INFO] Initialising MediaPipe FaceMesh...")

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,          # driver = single face
    refine_landmarks=True,    # better eye/iris precision
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

print("[INFO] FaceMesh ready.")


# ─────────────────────────────────────────────────────────────
#  OPEN WEBCAM
# ─────────────────────────────────────────────────────────────

cap = cv2.VideoCapture(0)   # change to 1 or 2 if wrong camera

if not cap.isOpened():
    print("[ERROR] Cannot open webcam. Try changing VideoCapture index.")
    sys.exit(1)

print("[INFO] Webcam opened. Press 'q' to quit.")


# ─────────────────────────────────────────────────────────────
#  STATE VARIABLES
# ─────────────────────────────────────────────────────────────

frame_counter = 0      # consecutive frames with EAR < threshold
alarm_on      = False  # is the alarm currently active?
alarm_thread  = None   # reference to alarm thread

session_start   = time.time()
total_alerts    = 0
alert_logged    = False  # avoid double-counting one alert event


# ─────────────────────────────────────────────────────────────
#  MAIN LOOP
# ─────────────────────────────────────────────────────────────

while True:
    ret, frame = cap.read()
    if not ret:
        print("[ERROR] Frame grab failed.")
        break

    # Flip for mirror effect (natural for front camera)
    frame = cv2.flip(frame, 1)
    h, w  = frame.shape[:2]

    # ── Step 1: BGR → RGB  (MediaPipe requires RGB) ──
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # ── Step 2: Run MediaPipe FaceMesh ──
    rgb.flags.writeable = False          # performance optimisation
    results = face_mesh.process(rgb)
    rgb.flags.writeable = True

    ear_avg = 0.0   # will update if face found

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            lm = face_landmarks.landmark

            # ── Step 3: Extract eye landmarks & compute EAR ──
            left_ear,  left_pts  = compute_ear(lm, LEFT_EYE,  w, h)
            right_ear, right_pts = compute_ear(lm, RIGHT_EYE, w, h)

            # Average EAR of both eyes
            ear_avg = (left_ear + right_ear) / 2.0

            # Draw eye outlines
            open_color  = (0, 255, 0)    # green  = open
            close_color = (0, 0, 255)    # red    = closed
            eye_color   = open_color if ear_avg > EAR_THRESHOLD else close_color
            draw_eye_outline(frame, left_pts,  eye_color)
            draw_eye_outline(frame, right_pts, eye_color)

            # ── Step 4: Frame Counter Logic ──
            # IF EAR < 0.25 → increment counter
            if ear_avg < EAR_THRESHOLD:
                frame_counter += 1

                # IF counter ≥ 15 frames → DROWSINESS ALERT
                if frame_counter >= FRAME_LIMIT:
                    if not alarm_on:
                        alarm_on = True
                        alert_logged = False

                    # Play alarm in separate thread (non-blocking)
                    if not alarm_on or (alarm_thread is None or not alarm_thread.is_alive()):
                        alarm_thread = threading.Thread(target=sound_alarm, daemon=True)
                        alarm_thread.start()

                    if not alert_logged:
                        total_alerts += 1
                        alert_logged  = True

                    # Big red alert banner
                    cv2.rectangle(frame, (0, 0), (w, 60), (0, 0, 200), -1)
                    cv2.putText(frame, "DROWSINESS ALERT! WAKE UP!",
                                (int(w * 0.12), 42),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)

                    # Red border around entire frame
                    cv2.rectangle(frame, (0, 0), (w - 1, h - 1), (0, 0, 255), 4)

                else:
                    # Warning zone — eyes starting to close
                    cv2.putText(frame, "Eyes closing...",
                                (15, 35), cv2.FONT_HERSHEY_SIMPLEX,
                                0.75, (0, 165, 255), 2)

            else:
                # IF EAR ≥ 0.25 → reset counter and alarm flag
                frame_counter = 0
                alarm_on      = False
                alert_logged  = False

    else:
        # No face detected
        cv2.putText(frame, "No face detected — check lighting",
                    (15, 35), cv2.FONT_HERSHEY_SIMPLEX,
                    0.65, (0, 200, 255), 2)

    # ── HUD overlay ──
    draw_status_bar(frame, ear_avg, frame_counter, alarm_on)

    # Session timer + alert count (top-right)
    elapsed     = int(time.time() - session_start)
    mins, secs  = divmod(elapsed, 60)
    cv2.putText(frame, f"{mins:02d}:{secs:02d}  Alerts: {total_alerts}",
                (w - 200, 25), cv2.FONT_HERSHEY_SIMPLEX,
                0.55, (200, 200, 200), 1)

    # ── Show frame ──
    cv2.imshow("Driver Drowsiness Detection — MediaPipe", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


# ─────────────────────────────────────────────────────────────
#  CLEANUP
# ─────────────────────────────────────────────────────────────

cap.release()
face_mesh.close()
cv2.destroyAllWindows()

elapsed    = int(time.time() - session_start)
mins, secs = divmod(elapsed, 60)
print("\n── Session Summary ──")
print(f"  Duration      : {mins:02d}:{secs:02d}")
print(f"  Total alerts  : {total_alerts}")
print("Done.")